In [115]:
print(1)

1


# Simple RAG Pipeline

## Data Ingestion

In [116]:
# dataloaders
from langchain.document_loaders import TextLoader
from langchain.document_loaders import DirectoryLoader


In [117]:
loader = TextLoader("sample.txt")
text_doc = loader.load()
print(text_doc) 

[Document(metadata={'source': 'sample.txt'}, page_content='Research Journal of Social Sciences & Economics Review  \nVol. 1, Issue 4, 2020 (October – December)                                                                                                                                                                                                                             \nISSN 2707-9023 (online), ISSN 2707-9015 (Print)                                       \nISSN 2707-9015 (ISSN-L) \nDOI: https://doi.org/10.36902/rjsser-vol1-iss4-2020(34-44)                                                            \nRJSSER \nResearch Journal of Social \nSciences & Economics Review \n____________________________________________________________________________________ \nImran Khan’s Speech at UNGA: A Reflection on Us vs. Them Divide Using \nFairclough’s 3D Model in CDA \n* Kinza Tariq \n** Shawal Muhammad Nawaz \n*** Dr. Aisha Farid, Assistant Professor (Corresponding Author) \n_________________

In [118]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Loading

#### Web Reader 

In [119]:
from langchain.document_loaders import WebBaseLoader
import bs4

In [120]:
loader = WebBaseLoader(web_path=("http://abdullaharifx.github.io/" ),
                       bs_kwargs={"parse_only": bs4.SoupStrainer(class_ = "main-content",)})
web_doc = loader.load()
web_doc[0].page_content

"\n\n\n\n\nAbout\n\n\nResume\n\n\nPortfolio\n\n\nBlog\n\n\nContact\n\n\n\n\n\n\nAbout me\n\n\nHow can a machine imitate a human? Can we fully achieve AGI by 2030?  \n\n         I’m Abdullah Arif, finding answers to these questions. Currently a Data Science trailblazer at FAST-NUCES, chasing the electric pulse of AI where algorithms dance with human dreams. I don’t just write code—I sculpt solutions, blending logic with wild imagination to make tech sing.\n        \n\n            My journey? It’s a whirlwind of data, creativity, and a sprinkle of chaos. From crafting Urdu voice schedulers that hum with cultural rhythm to building lip-reading systems that amplify silent voices in Pakistan, I wield Python, TensorFlow, and PySpark to turn messy data into dazzling insights.\n          \n\n\n\nWhat I'm doing\n\n\n\n\n\n\nCustom AI/ML Solutions\n\n                  Unlock the power of AI with tailored ML models for predictive analytics, automation, and decision-making. I build scalable soluti

#### PDF Reader

In [121]:

from langchain.document_loaders import PyPDFLoader
loader = PyPDFLoader("sample.pdf")
pdf_doc = loader.load()
pdf_doc[0].page_content

'National University of Computer and Emerging Sciences  \n  \n  \nPage 1 of 6  \n  \nData Mining (DS3002)  \nDate: April 3rd 2024  \nCourse Instructor  \nMiss Eesha Tur Razia Babar  \n  \n  \n  \n  \n  \n  \nSessional-I Exam  \nTotal Time: 1 Hours Total \nMarks: 25   \nTotal Questions: 05  \n  \nSemester: Spring-2024  \nCampus: Lahore  \nDept: AI and Data Science  \n____________________________   _______   _______   _____________________  \nStudent Name                                                 Roll No          Section       Student Signature  \n  \n  \n____________________________                                       _____________________  \nVetted by                                                                                                    Vetter Signature  \n  \n \n  \nCLO 1:    Understand basic concepts of data mining.                                                       \n \nQ1: Suppose that we have training data f(x(1); y(1)); (x(2); y(2)); : : : ; (x(m); y(m)) an

### Transform

In [166]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
test_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
# for splitting documents
doc = test_splitter.split_documents(web_doc + pdf_doc + text_doc)
doc[0].page_content 

"About\n\n\nResume\n\n\nPortfolio\n\n\nBlog\n\n\nContact\n\n\n\n\n\n\nAbout me\n\n\nHow can a machine imitate a human? Can we fully achieve AGI by 2030?  \n\n         I’m Abdullah Arif, finding answers to these questions. Currently a Data Science trailblazer at FAST-NUCES, chasing the electric pulse of AI where algorithms dance with human dreams. I don’t just write code—I sculpt solutions, blending logic with wild imagination to make tech sing.\n        \n\n            My journey? It’s a whirlwind of data, creativity, and a sprinkle of chaos. From crafting Urdu voice schedulers that hum with cultural rhythm to building lip-reading systems that amplify silent voices in Pakistan, I wield Python, TensorFlow, and PySpark to turn messy data into dazzling insights.\n          \n\n\n\nWhat I'm doing\n\n\n\n\n\n\nCustom AI/ML Solutions\n\n                  Unlock the power of AI with tailored ML models for predictive analytics, automation, and decision-making. I build scalable solutions using 

### Vector Embedding


In [196]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model = "models/embedding-001")
len(embeddings.embed_query("hi"))

768

#### DB for Vectors

In [ ]:
from langchain_community.vectorstores import Chroma, FAISS
db = Chroma.from_documents(doc, embeddings)
db2 = FAISS.from_documents(doc, embeddings) 

### Retrieval of Docs

In [198]:
query = "UNGA"
results = db.similarity_search_with_score(query, k=3)
filtered = [doc for doc, score in results if score > 0.7]  # or your own threshold
print(filtered)
res = db.similarity_search(query, k=3)
res2 = db2.similarity_search(query, k=3)

[]


In [170]:
res2

[Document(id='4398d644-60ed-483c-a083-05fc466889f2', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-04-15T12:32:29+05:00', 'author': 'Tanzil Rehman', 'moddate': '2024-04-15T12:32:29+05:00', 'source': 'sample.pdf', 'total_pages': 6, 'page': 5, 'page_label': '6'}, page_content='National University of Computer and Emerging Sciences  \n  \n  \nPage 6 of 6'),
 Document(id='75dfbfba-de20-41a5-90fb-cf5e9e3c716d', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-04-15T12:32:29+05:00', 'author': 'Tanzil Rehman', 'moddate': '2024-04-15T12:32:29+05:00', 'source': 'sample.pdf', 'total_pages': 6, 'page': 2, 'page_label': '3'}, page_content='National University of Computer and Emerging Sciences  \n  \n  \nPage 3 of 6  \n  \n  \n  \n 2 marks'),
 Document(id='670432c0-c33d-4e07-abe0-cb410ae18484', metadata={'source': 'http://abdullaharifx.git

In [192]:
res  = db.similarity_search_with_score("UNGA", k=3)
res


[(Document(metadata={'moddate': '2024-04-15T12:32:29+05:00', 'page_label': '6', 'author': 'Tanzil Rehman', 'producer': 'Microsoft® Word for Microsoft 365', 'page': 5, 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-04-15T12:32:29+05:00', 'total_pages': 6, 'source': 'sample.pdf'}, page_content='National University of Computer and Emerging Sciences  \n  \n  \nPage 6 of 6'),
  0.2675398886203766),
 (Document(metadata={'page_label': '6', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-04-15T12:32:29+05:00', 'page': 5, 'moddate': '2024-04-15T12:32:29+05:00', 'total_pages': 6, 'author': 'Tanzil Rehman', 'source': 'sample.pdf', 'producer': 'Microsoft® Word for Microsoft 365'}, page_content='National University of Computer and Emerging Sciences  \n  \n  \nPage 6 of 6'),
  0.2675398886203766),
 (Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'moddate': '2024-04-15T12:32:29+05:00', 't

In [194]:
# for i, d in enumerate(doc):
#     if "unga" in d.page_content.lower() or "united nations" in d.page_content.lower():
#         print(f"[{i}] -> {d.page_content}")